# Skull acoustic transparency at **500 kHz** — Colab

Time-reversal skull *transparency maps* + transducer placement, end to end, on a real benchmark skull, small enough for a free Colab GPU. A **manuscript-style figure at every step**.

**Pipeline:** `prepare` (skull map → solver inputs) → `sim outward --run` (full-wave GPU solve) → `extract` (→ Field Bundle) → `transparency` / `place`.

**Skull:** the ITRUSST transcranial-ultrasound benchmark skull (Aubry et al., *JASA* 2022), which is defined at 500 kHz. Its surface meshes are rasterized to homogeneous cortical bone (`c=2800`, water `1500`).

**Grid & runtime.** `dx = c0 / (f0·ppw)`, run at **6 PPW (~0.5 mm)** — benchmark-grade. The outward solve uses the **surface-shell recorder** (`--recorder shell`): it records the field only on the calvarial surface (all the transparency map needs), so there's **no giant `genout_mod` dump** — `genout.dat` stays sub-GB and `extract` no longer OOMs, even for the whole skull at 6 PPW. The one remaining limit is the **solve's GPU memory** (~14 GB for the whole skull at 6 PPW): comfortable on Colab's **L4 / A100** (Runtime ▸ Change runtime type). A **T4 (16 GB)** is borderline — if the solve OOMs the GPU, switch to an **L4/A100** runtime; the resolution stays at 6 PPW.

## 0 · Runtime

**Runtime ▸ Change runtime type ▸ GPU (T4)**, then run the cells top to bottom.

In [ ]:
# --- setup: install the package + fetch the CUDA solver binary + mesh tooling ---
import os, sys, glob, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules

# skull_transparency + trimesh (rasterize the skull STL).
!pip -q install "git+https://github.com/pinton-lab/skull_transparency.git" trimesh

# The full-wave CUDA solver binary ships in the fullwave2-ultra repo.
# Clone it and point FULLWAVE2_BIN at it.
!git clone --depth 1 https://github.com/pinton-lab/fullwave2-ultra.git /content/fullwave2-ultra 2>/dev/null || true
hits = glob.glob("/content/fullwave2-ultra/**/bench_3d_opt", recursive=True)   # don't hardcode the path
assert hits, "bench_3d_opt not found in the clone -- re-run this cell (network?) or check the repo."
os.environ["FULLWAVE2_BIN"] = hits[0]
os.chmod(hits[0], 0o755)                                                       # git may drop the +x bit
print("solver:", os.environ["FULLWAVE2_BIN"], "| executable:", os.access(hits[0], os.X_OK))

# GPU report -- the Section-2 solve needs ~14 GB (6 PPW whole skull). L4/A100 comfortable; T4 borderline.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print("GPU:", gpu.stdout.strip(), "\\n(need ~14 GB for the 6-PPW whole-skull solve; if <16 GB, prefer L4/A100)")
else:
    print("NO GPU visible -- Runtime > Change runtime type > GPU (L4/A100 preferred) before Section 2.")

## Figure helpers (manuscript style)

One reusable set of multi-panel figures, used at each step below. They work on both the no-GPU synthetic bundle (Section 1) and the real solved bundle (Section 2).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

INK, EDGE, FOCUS = "#1d1d1f", "#2c6fbf", "#c0392b"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "figure.facecolor": "white",
                     "axes.facecolor": "white", "axes.titlesize": 10, "savefig.dpi": 120})

def _ortho(ax3, vol, dx, cmap, vmin, vmax, log=False, titles=("sagittal","coronal","axial")):
    v = np.asarray(vol, float); ctr = np.array(v.shape) // 2
    d = np.log10(np.maximum(v, (v[v>0].min() if (v>0).any() else 1))) if log else v
    lo = np.log10(vmin) if (log and vmin>0) else vmin
    hi = np.log10(vmax) if (log and vmax>0) else vmax
    for a, (s, t) in zip(ax3, zip([d[ctr[0]].T, d[:,ctr[1]].T, d[:,:,ctr[2]].T], titles)):
        im = a.imshow(s, origin="lower", cmap=cmap, vmin=lo, vmax=hi, aspect="equal",
                      extent=[0, s.shape[1]*dx, 0, s.shape[0]*dx])
        a.set_title(t); a.set_xlabel("mm")
    ax3[0].set_ylabel("mm"); return im

def fig_skull(c, dx, title="Skull sound-speed map  c(x)"):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
    im = _ortho(ax, c, dx, "bone", 1500, float(np.max(c)))
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="c (m/s)")
    fig.suptitle(title, fontweight="bold"); plt.show()


# -- volumetric overlays: the transparency / footprint sit ON the skull cross-section (like the manuscripts) --
_PLANES = [(0, 1, 2, "sagittal"), (1, 0, 2, "coronal"), (2, 0, 1, "axial")]   # (slice_axis, x_axis, y_axis)

def _center_vox(t, dx):
    # brain-center source: each surface patch lies at source + rhat*(radius/dx) along its ray
    src = np.asarray(t.surf_vox) - np.asarray(t.rhat) * (np.asarray(t.rad_mm)[:, None] / dx)
    return np.round(np.median(src, axis=0)).astype(int)

def _ortho_bg(ax3, c, dx, ctr):
    v = np.asarray(c, float); ctr = np.clip(np.asarray(ctr, int), 0, np.array(v.shape) - 1)
    for a, (sa, xa, ya, ttl) in zip(ax3, _PLANES):
        sl = [v[ctr[0]], v[:, ctr[1]], v[:, :, ctr[2]]][sa].T
        a.imshow(sl, origin="lower", cmap="gray", vmin=float(v.min()), vmax=float(v.max()), aspect="equal",
                 extent=[0, sl.shape[1]*dx, 0, sl.shape[0]*dx])
        a.set_title(ttl); a.set_xticks([]); a.set_yticks([])
    return ctr

def _overlay(ax3, pts, ctr, dx, slab, c=None, cmap=None, **kw):
    P = np.asarray(pts, float); sc = None
    for a, (sa, xa, ya, _) in zip(ax3, _PLANES):
        m = np.abs(P[:, sa] - ctr[sa]) <= slab            # patches near this slice plane
        sc = a.scatter(P[m, xa]*dx, P[m, ya]*dx, c=(np.asarray(c)[m] if c is not None else FOCUS),
                       cmap=cmap, **kw)
    return sc

def fig_surface(t, c, dx, vals, title, cmap="inferno", label="value"):
    # per-patch surface quantity (e.g. peak pressure) painted on the skull cross-section
    ctr = _center_vox(t, dx); v = np.asarray(vals, float); vn = v / (np.max(v) or 1)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.4), constrained_layout=True)
    _ortho_bg(ax, c, dx, ctr)
    sc = _overlay(ax, t.surf_vox, ctr, dx, slab=4, c=vn, cmap=cmap, s=10, vmin=0, vmax=1, linewidths=0)
    fig.colorbar(sc, ax=ax, fraction=0.02, label=label)
    fig.suptitle(title, fontsize=12, fontweight="bold"); plt.show()

def fig_transparency(t, c, dx, title="Skull transparency map"):
    surf = np.asarray(t.surf_vox); vn = np.maximum(t.value, 0) / (np.max(t.value) or 1)
    ctr = _center_vox(t, dx)
    fig = plt.figure(figsize=(13, 8), constrained_layout=True); gs = fig.add_gridspec(2, 3)
    top = [fig.add_subplot(gs[0, i]) for i in range(3)]
    _ortho_bg(top, c, dx, ctr)                            # skull cross-section (grayscale)
    sc = _overlay(top, surf, ctr, dx, slab=4, c=vn, cmap="inferno", s=10, vmin=0, vmax=1, linewidths=0)
    fig.colorbar(sc, ax=top, fraction=0.02, label="normalized transparency  (bone cross-section)")
    b0 = fig.add_subplot(gs[1, 0]); b0.hist(vn, bins=40, color=EDGE, alpha=0.85)
    b0.set_title("transparency histogram"); b0.set_xlabel("normalized"); b0.set_ylabel("patches")
    b1 = fig.add_subplot(gs[1, 1]); b1.scatter(np.asarray(t.rad_mm), vn, s=5, color=EDGE, alpha=0.4)
    b1.set_title("transparency vs source distance"); b1.set_xlabel("distance (mm)"); b1.set_ylabel("normalized")
    b2 = fig.add_subplot(gs[1, 2]); b2.axis("off")
    b2.text(0.5, 0.55, f"{len(surf)} surface patches\nbrighter = more transparent bone\n\n"
            f"source depth {np.median(t.rad_mm):.0f} mm (median)", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.6", fc="#f4f6f9", ec=EDGE))
    fig.suptitle(title, fontsize=12, fontweight="bold"); plt.show()

def fig_placement(t, pl, c, dx, title="Transducer placement"):
    surf = np.asarray(t.surf_vox); foot = np.zeros(len(surf), bool); foot[np.asarray(pl.footprint_surf_idx)] = True
    ctr = _center_vox(t, dx)
    fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.7), constrained_layout=True)
    _ortho_bg(ax, c, dx, ctr)                             # skull cross-section (grayscale)
    _overlay(ax, surf[~foot], ctr, dx, slab=4, s=5, alpha=0.35)               # skull surface (context)
    _overlay(ax, surf[foot], ctr, dx, slab=8, s=24, alpha=0.95)              # bowl footprint (red)
    for a, (sa, xa, ya, _) in zip(ax, _PLANES):
        a.plot(ctr[xa]*dx, ctr[ya]*dx, "x", ms=13, mew=2.5, color="#2ca02c")  # target / source
    fig.suptitle(f"{title}   —   footprint {pl.n_footprint_patches} patches, incidence "
                 f"{pl.incidence_deg:.1f} deg, transparency score {pl.transparency_score:.3f}",
                 fontsize=12, fontweight="bold"); plt.show()

print("figure helpers ready")

## 1 · Quick taste — no GPU, no data

The analysis + figures on a shipped **synthetic** Field Bundle, so you see the whole thing before the GPU solve.

In [ ]:
import skull_transparency as st
b0 = st.load_bundle(st.make_synthetic_bundle("synthetic_bundle"))
t0 = st.compute_transparency_map(b0)
pl0 = st.place_bowl(t0, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=15.0, theta_max_deg=35.0))
print(f"{len(t0.surf_vox)} surface patches | placement incidence {pl0.incidence_deg:.1f} deg, "
      f"score {pl0.transparency_score:.3f}")
dx0 = b0.grid["dx_m"] * 1e3
fig_transparency(t0, b0.skull_c(), dx0, title="Transparency map (synthetic, no GPU)")
fig_placement(t0, pl0, b0.skull_c(), dx0, title="Placement (synthetic, no GPU)")

## 2 · Real 500 kHz simulation on the ITRUSST benchmark skull

### 2a · Build the skull sound-speed map  — *figure: input skull*

Downloads the benchmark's inner/outer surface meshes (~13 MB) and rasterizes them to the grid as homogeneous cortical bone at **6 PPW (~0.5 mm)**. Set `USE_REAL_SKULL = False` for an offline toy shell.

In [ ]:
import os, urllib.request, numpy as np

PPW = 6                        # 6 PPW (~0.5 mm) = benchmark-grade (do not lower). The shell recorder
                               # keeps the OUTPUT sub-GB; the solve needs ~14 GB GPU at 6 PPW whole-skull
                               # -> use an L4/A100 runtime (a T4 is borderline; USE_REAL_SKULL=False fits it).
USE_REAL_SKULL = True
C0, F0 = 1500.0, 500e3
pitch = round(C0 / (F0 * PPW) * 1e3, 3)     # mm/voxel to match the sim grid

if USE_REAL_SKULL:
    import trimesh
    BASE = ("https://raw.githubusercontent.com/ucl-bug/transcranial-ultrasound-benchmarks"
            "/master/intercomparison/skull-stl")
    for f in ("skull_inner.stl", "skull_outer.stl"):
        if not os.path.exists(f):
            urllib.request.urlretrieve(f"{BASE}/{f}", f)
    outer = trimesh.load("skull_outer.stl"); inner = trimesh.load("skull_inner.stl")
    lo = outer.bounds[0] - 2*pitch
    dims = np.ceil((outer.bounds[1] + 2*pitch - lo) / pitch).astype(int)
    def _raster(mesh):
        vg = mesh.voxelized(pitch=pitch).fill(); idx = np.floor((vg.points - lo) / pitch).astype(int)
        g = np.zeros(dims, bool); ok = np.all((idx >= 0) & (idx < dims), axis=1); idx = idx[ok]
        g[idx[:,0], idx[:,1], idx[:,2]] = True; return g
    bone = _raster(outer) & ~_raster(inner)
    c = np.where(bone, 2800.0, C0).astype(np.float32)
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = lo
    print(f"ITRUSST skull @ {pitch} mm: {tuple(int(d) for d in dims)} = {c.size/1e6:.1f} M voxels; "
          f"bone {bone.sum()*pitch**3/1000:.0f} cm^3")
else:
    n = int(round(120 * 0.5 / pitch))
    c = np.full((n, n, n), C0, np.float32)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n]
    r = np.sqrt((xx-n/2)**2 + (yy-n/2)**2 + (zz-n/2)**2) * pitch
    c[(r > 40) & (r < 46)] = 2600.0
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = -pitch * n / 2
    print("toy shell:", c.shape)

np.save("head_c.npy", c); np.save("affine.npy", affine)
import json; json.dump({"preset": "ctx500", "f0_hz": F0, "ppw": PPW}, open("ctx500.json", "w"))
fig_skull(c, pitch, title=f"Input skull @ {pitch} mm  ({'ITRUSST benchmark' if USE_REAL_SKULL else 'toy shell'})")

### 2b · `prepare` — skull map → solver inputs (no GPU)

Seats an omnidirectional source at the brain center (`--center`) for the whole-skull transparency map, and writes the `.dat` solver deck at `dx = c0/(f0·ppw)`.

In [ ]:
!skull-transparency prepare --c-map head_c.npy --affine affine.npy --center --transducer ctx500.json --out run
import json; m = json.load(open("run/meta.json"))
print(f"sim grid N={m['N']}  dx={m['dX_m']*1e3:.3f} mm  F0={m['F0']/1e3:.0f} kHz")

### 2c · Outward full-wave solve (GPU)

Runs the CUDA solver with `--recorder shell` — it records the field only on the calvarial surface (all the transparency map needs), so there is **no** `genout_mod` volume dump and `genout.dat` stays sub-GB. A few minutes on a GPU at 6 PPW. *(The field figures come after `extract`, in 2d.)*

In [ ]:
import os, sys, subprocess
# capture the solver output so a GPU-off / out-of-memory failure is visible and diagnosed here
r = subprocess.run([sys.executable, "-m", "skull_transparency.sim", "outward",
                    "--sim", "run", "--out", "run", "--run", "--recorder", "shell"],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
go = "run/outward/genout.dat"
if r.returncode != 0 or not (os.path.exists(go) and os.path.getsize(go) > 0):
    print("---- solver stderr (tail) ----\\n" + r.stderr[-3000:])
    raise RuntimeError(
        "Outward solve failed -- see the solver output above. Most common causes:\\n"
        "  - GPU runtime OFF: Runtime > Change runtime type > GPU, then re-run from Section 0.\\n"
        "  - CUDA out-of-memory: the whole skull at 6 PPW needs ~14 GB -> switch to an L4/A100\\n"
        "    runtime (Runtime > Change runtime type). Keep PPW=6; do not lower it.\\n"
        "  - solver binary missing: re-run the setup cell in Section 0.")
assert not os.path.exists("run/outward/genout_mod.dat"), \
    "unexpected genout_mod.dat (the shell recorder should not write one)"
print(f"OK -- genout.dat (surface shell): {os.path.getsize(go)/1e6:.0f} MB   |   genout_mod: none")

### 2d · Extract → Field Bundle → transparency map  — *figures: outward surface field + transparency*

The shell recorder captures the field **on the calvarial surface** (no volume dump), so the field figures are surface quantities shown on the skull cross-section: first the raw peak pressure reaching the bone, then the transparency map derived from it.

In [ ]:
!skull-transparency extract --run run/outward --sim run --out run/bundle

import skull_transparency as st
b = st.load_bundle("run/bundle")
c, dx = b.skull_c(), b.grid["dx_m"]*1e3
t = st.compute_transparency_map(b)

fig_surface(t, c, dx, t.Pmax, title="Outward field @ 500 kHz — peak pressure at the calvaria (ITRUSST skull)",
            cmap="inferno", label="normalized peak pressure")
fig_transparency(t, c, dx, title="Skull transparency @ 500 kHz  (ITRUSST benchmark skull)")

### 2e · Place a focused window on the map  — *figure: placement*

In [ ]:
pl = st.place_bowl(t, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=32.0, theta_max_deg=35.0))
print(f"incidence {pl.incidence_deg:.1f} deg, transparency score {pl.transparency_score:.3f}")
fig_placement(t, pl, c, dx, title="Transducer placement on the transparency map")